# 🧬 Bio-JEPA — Phase 5 : Approfondissement Expérimental
**Ablation Study · EGFR (2ème cible) · Comparaison AutoDock Vina**

---

## ⚙️ Avant de commencer
1. `Runtime → Change runtime type → A100 GPU`
2. Avoir les checkpoints sur Google Drive depuis la Phase 1/2
3. Exécuter les cellules dans l'ordre

---

## 📋 Plan
| Étape | Description | Durée estimée |
|---|---|---|
| 0 | GPU + Google Drive + Setup | 5 min |
| 1 | Ablation Study (EMA, stop-grad, masquage) | ~2h |
| 2 | Deuxième cible : EGFR (CHEMBL203) | ~3h |
| 3 | Benchmark AutoDock Vina vs Bio-JEPA | ~1h |
| 4 | Sauvegarde finale sur Drive | 2 min |

---
## Étape 0 — Setup

In [2]:
import torch
!nvidia-smi
print(f'\n✓ GPU : {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Sat Mar  7 20:40:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/Bio-JEPA-checkpoints'
DRIVE_RES  = '/content/drive/MyDrive/Bio-JEPA-results'
os.makedirs(DRIVE_RES, exist_ok=True)

print(f'✓ Drive monté')
print(f'✓ Checkpoints : {DRIVE_CKPT}')
!ls -lh {DRIVE_CKPT}

Mounted at /content/drive
✓ Drive monté
✓ Checkpoints : /content/drive/MyDrive/Bio-JEPA-checkpoints
total 25M
-rw------- 1 root root  13M Mar  6 13:02 best_model.pt
-rw------- 1 root root  64K Mar  6 13:03 few_shot_curve.png
-rw------- 1 root root 1.3K Mar  6 13:02 few_shot_indomain.json
-rw------- 1 root root 1.3K Mar  6 13:02 few_shot_transfer.json
-rw------- 1 root root  13M Mar  6 11:18 zinc_pretrained.pt


In [4]:
!pip install torch_geometric rdkit pandas numpy scikit-learn tqdm requests pyyaml chembl-webresource-client -q
print('✓ Dépendances installées')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.4 MB/s eta 0:00:00
✓ Dépendances installées


In [23]:
!git -C /content/Bio-JEPA pull origin main

From https://github.com/7Nayy/Bio-JEPA
 * branch            main       -> FETCH_HEAD
Already up to date.


In [5]:
!git clone https://github.com/7Nayy/Bio-JEPA.git /content/Bio-JEPA
os.chdir('/content/Bio-JEPA')
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)
shutil.copy(f'{DRIVE_CKPT}/zinc_pretrained.pt', 'checkpoints/zinc_pretrained.pt')
shutil.copy(f'{DRIVE_CKPT}/best_model.pt', 'checkpoints/best_model.pt')
print('✓ Repo cloné + checkpoints récupérés')
!ls checkpoints/

Cloning into '/content/Bio-JEPA'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 102 (delta 8), reused 36 (delta 6), pack-reused 54 (from 2)
Receiving objects: 100% (102/102), 48.13 MiB | 15.52 MiB/s, done.
Resolving deltas: 100% (11/11), done.
✓ Repo cloné + checkpoints récupérés
best_model.pt  final_model.pt  probe_chembl251.pt  zinc_pretrained.pt


---
## Étape 1 — Ablation Study

> **Objectif** : Prouver que chaque composant de Bio-JEPA contribue aux performances  
> **4 variantes** : complet / sans EMA / sans masquage progressif / sans stop-gradient  
> **Durée estimée** : ~2h sur A100  
> ⚠️ Ne pas interrompre cette cellule

In [7]:
!python ablation_study.py \
    --checkpoint checkpoints/zinc_pretrained.pt \
    --target CHEMBL251 \
    --runs 3 \
    --out-dir results/

[Dispositif] cuda
[Checkpoint] checkpoints/zinc_pretrained.pt

[Données] ChEMBL — target=CHEMBL251
  Train : 6,739  |  Test : 843

[Plan] 4 variantes × 3 runs × 50 epochs + 100 epochs sonde
       Estimé : ~190 min (approximatif, CPU/GPU)

────────────────────────────────────────────────────────────
  V1 — Bio-JEPA complet (référence)
────────────────────────────────────────────────────────────

    [Run 1/3  seed=0]
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
      Epoch  10/50  loss=0.0005  mask=0.12
      Epoch  20/50  loss=0.0005  mask=0.17
      Epoch  30/50  loss=0.0005  mask=0.23
      Epoch  40/50  loss=0.0005  mask=0.28
      Epoch  50/50  loss=0.0005  mask=0.30
      → Pearson r=0.7419  Spearman ρ=0.7448  RMSE=1.9363  (217s)

    [Run 2/3  seed=1]
      Epoch  10/50  loss=0.0005  mask=0.12
      Epoch  20

In [10]:
import json, pandas as pd

with open('results/ablation_results.json') as f:
    data = json.load(f)

rows = []
for key, val in data['variantes'].items():
    rows.append({
        'Variante': val['nom'],
        'Pearson r': f"{val['metriques']['pearson_r']['mean']:.3f} ± {val['metriques']['pearson_r']['std']:.3f}",
        'RMSE': f"{val['metriques']['rmse']['mean']:.3f}",
    })

df = pd.DataFrame(rows)
print('=== ABLATION STUDY — Bio-JEPA sur ChEMBL251 ===')
print(df.to_string(index=False))

=== ABLATION STUDY — Bio-JEPA sur ChEMBL251 ===
                      Variante     Pearson r  RMSE
  Bio-JEPA complet (référence) 0.739 ± 0.005 1.929
  Sans EMA (copie directe τ=0) 0.718 ± 0.004 1.913
Sans masquage progressif (30%) 0.728 ± 0.006 1.928
            Sans stop-gradient 0.517 ± 0.027 1.740


---
## Étape 2 — Deuxième cible biologique : EGFR (CHEMBL203)

> **Objectif** : Démontrer que Bio-JEPA généralise au-delà de A2A  
> **Cible** : EGFR — récepteur tyrosine kinase impliqué dans le cancer du poumon  
> **Durée estimée** : ~3h sur A100

In [6]:
# Few-shot evaluation sur EGFR avec le checkpoint ZINC (transfert)
!python few_shot_eval.py \
    --checkpoint checkpoints/zinc_pretrained.pt \
    --target CHEMBL203 \
    --n-values 10,50,100,200,500,1000 \
    --n-runs 5 \
    --save-json results/few_shot_egfr.json

  Évaluation Few-Shot : Bio-JEPA vs GNN supervisé
  Cible : CHEMBL203 — EGFR (kinase)
  N ∈ [10, 50, 100, 200, 500, 1000]
  5 runs par N  │  Dispositif : cuda

[Données] Chargement du dataset ChEMBL — target=CHEMBL203
  Matérialisation du train set en liste...

  Train total : 6,739  │  Val : 842  │  Test : 843

[Bio-JEPA] Checkpoint : checkpoints/zinc_pretrained.pt

[Bio-JEPA] Extraction des embeddings (Target Encoder figé)...
  Train : (6739, 256)  │  Val : (842, 256)  │  Test : (843, 256)  │  1.5s

──────────────────────────────────────────────────────────────────────
  Démarrage des expériences (6 valeurs de N × 5 runs)
  Bio-JEPA : 200 époques (sonde)  │  GNN sup. : 300 époques
──────────────────────────────────────────────────────────────────────

  ── N = 10 ──────────────────────────────────────────────
    Bio-JEPA  N=   10  run 1/5  →  r=-0.1308  ρ=-0.1098  RMSE=3.8538
    Bio-JEPA  N=   10  run 2/5  →  r=-0.0032  ρ=+0.0575  RMSE=3.7010
    Bio-JEPA  N=   10  run 3/5  →  r=+0

In [17]:
import json, pandas as pd, shutil

shutil.copy(f'{DRIVE_RES}/few_shot_transfer.json', 'results/few_shot_transfer.json')

with open('results/few_shot_transfer.json') as f:
    a2a = json.load(f)
with open('results/few_shot_egfr.json') as f:
    egfr = json.load(f)

N = a2a['N_values']
rows = []
for i, n in enumerate(N):
    rows.append({
        'N': n,
        'Bio-JEPA A2A':  round(a2a['bio_jepa'][i]['mean_r'], 3),
        'Bio-JEPA EGFR': round(egfr['bio_jepa'][i]['pearson_r']['mean'], 3),
        'GNN sup. A2A':  round(a2a['gnn_supervise'][i]['mean_r'], 3),
        'GNN sup. EGFR': round(egfr['gnn_supervise'][i]['pearson_r']['mean'], 3),
    })

df = pd.DataFrame(rows)
print('=== FEW-SHOT : A2A vs EGFR — Pearson r ===')
print(df.to_string(index=False))

shutil.copy('results/few_shot_egfr.json', f'{DRIVE_RES}/few_shot_egfr.json')
print('\n✓ Résultats EGFR sauvegardés sur Drive')

=== FEW-SHOT : A2A vs EGFR — Pearson r ===
   N  Bio-JEPA A2A  Bio-JEPA EGFR  GNN sup. A2A  GNN sup. EGFR
  10         0.036          0.003         0.047          0.037
  50         0.130          0.130         0.338          0.356
 100         0.174          0.174         0.465          0.467
 200         0.278          0.277         0.530          0.523
 500         0.465          0.465         0.651          0.651
1000         0.542          0.539         0.704          0.708

✓ Résultats EGFR sauvegardés sur Drive


---
## Étape 3 — Benchmark : Bio-JEPA vs AutoDock Vina

> **Objectif** : Quantifier l'avantage de vitesse de Bio-JEPA sur la référence du virtual screening  
> **Durée estimée** : ~1h sur A100

In [18]:
# Installation AutoDock Vina
!pip install vina -q
!wget -q https://files.rcsb.org/download/3EML.pdb -O data/3EML.pdb
print('✓ AutoDock Vina installé + structure A2A (3EML) téléchargée')

✓ AutoDock Vina installé + structure A2A (3EML) téléchargée


In [24]:
!python autodock_benchmark.py \
    --checkpoint checkpoints/zinc_pretrained.pt \
    --target CHEMBL251 \
    --n-mols 50 \
    --out results/autodock_benchmark.json

[Dispositif] cuda

[Modèle]
  checkpoints/zinc_pretrained.pt  (407,044 paramètres — target encoder)

[Données] CHEMBL251
  8,424 molécules

[Sonde MLP]
  Entraînement de la sonde MLP sur l'ensemble du dataset...
  Embeddings extraits : (8424, 256)
    epoch  30/150  val_loss=45.3425
    epoch  60/150  val_loss=33.7769
    epoch  90/150  val_loss=21.9221
    epoch 120/150  val_loss=19.6877
    epoch 150/150  val_loss=19.4126
  Sonde sauvegardée → checkpoints/probe_benchmark.pt

[Sélection] 50 molécules aléatoires (seed=42)
  pChEMBL : min=4.56  max=10.44  mean=7.28  std=1.35

[Bio-JEPA] Scoring de 50 molécules...
  Temps total : 4.2 ms  (0.084 ms/mol)
  Pearson r   : 0.592

[AutoDock Vina]
  Binaire vina introuvable (pip install vina échoue sur macOS ARM).
  → Mode simulation (estimations littérature).
  Temps estimé : 1.7 min  (2.0 s/mol)
  Pearson r    : 0.350  (littérature)

════════════════════════════════════════════════════════════════════════════
  TABLEAU COMPARATIF : Bio-JEPA v

In [27]:
import json, shutil

with open('results/autodock_benchmark.json') as f:
    bench = json.load(f)

bio = bench['bio_jepa']
auto = bench['autodock_vina']

print('=== BENCHMARK VITESSE : Bio-JEPA vs AutoDock Vina ===')
print(f"{'Méthode':<25} {'50 mols':>10} {'1 000 mols':>12} {'1M mols':>12} {'Pearson r':>10}")
print('-' * 72)

# Bio-JEPA
t50_bio = f"{bio['elapsed_s']*1000:.1f}ms"
t1k_bio = f"{bio['elapsed_s']*20*1000:.0f}ms"
t1M_bio = f"{bio['elapsed_s']*20000/60:.1f}min"
print(f"{'Bio-JEPA':<25} {t50_bio:>10} {t1k_bio:>12} {t1M_bio:>12} {bio['pearson_r']:>10.3f}")

# AutoDock
t50_auto = f"{auto['elapsed_s']:.1f}s"
t1k_auto = f"{auto['elapsed_s']*20/60:.1f}min"
t1M_auto = f"{auto['elapsed_s']*20000/3600:.1f}h"
print(f"{'AutoDock Vina (sim)':<25} {t50_auto:>10} {t1k_auto:>12} {t1M_auto:>12} {auto.get('pearson_r', 0.350):>10.3f}")

print(f"\n✓ Accélération Bio-JEPA vs AutoDock : x{auto['elapsed_s']/bio['elapsed_s']:.0f}")

shutil.copy('results/autodock_benchmark.json', f'{DRIVE_RES}/autodock_benchmark.json')
print('✓ Benchmark sauvegardé sur Drive')

=== BENCHMARK VITESSE : Bio-JEPA vs AutoDock Vina ===
Méthode                      50 mols   1 000 mols      1M mols  Pearson r
------------------------------------------------------------------------
Bio-JEPA                       4.2ms         84ms       1.4min      0.592
AutoDock Vina (sim)           100.0s      33.3min       555.6h      0.350

✓ Accélération Bio-JEPA vs AutoDock : x23864
✓ Benchmark sauvegardé sur Drive


---
## Étape 4 — Sauvegarde finale sur Google Drive

In [28]:
fichiers = {
    'results/ablation_results.json'    : 'ablation_results.json',
    'results/few_shot_egfr.json'       : 'few_shot_egfr.json',
    'results/autodock_benchmark.json'  : 'autodock_benchmark.json',
}

for src, dst in fichiers.items():
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_RES}/{dst}')
        size = os.path.getsize(f'{DRIVE_RES}/{dst}') / 1e3
        print(f'✓ {dst} ({size:.1f} KB)')
    else:
        print(f'⚠️  {src} introuvable — étape précédente non complétée')

print('\n✅ PHASE 5 TERMINÉE')
print(f'📁 Résultats dans : {DRIVE_RES}')

✓ ablation_results.json (2.5 KB)
✓ few_shot_egfr.json (10.7 KB)
✓ autodock_benchmark.json (7.1 KB)

✅ PHASE 5 TERMINÉE
📁 Résultats dans : /content/drive/MyDrive/Bio-JEPA-results
